In [1]:
import numpy as np
import pandas as pd

# Load the dataset (Assuming the file name is 'retail_sales.csv')
df = pd.read_csv("retail_sales.csv")

# ==========================================
# 1. Data Cleaning
# ==========================================

# --- Handle missing values ---
# Fill missing CustomerID values with "Unknown"
df["CustomerID"] = df["CustomerID"].fillna("Unknown")

# Drop rows where TotalAmount is missing
df = df.dropna(subset=["TotalAmount"])

# --- Remove duplicates ---
# Check for and remove duplicate rows
df = df.drop_duplicates()

# --- Fix incorrect data ---
# Replace negative Quantity values with 0 and adjust TotalAmount accordingly
# (Do this before checking for mismatches so TotalAmount matches the new Quantity)
df.loc[df["Quantity"] < 0, "TotalAmount"] = 0
df.loc[df["Quantity"] < 0, "Quantity"] = 0

# Identify and correct transactions where TotalAmount does not match Quantity * Price
# (We recalculate to fix incorrect entries)
df["TotalAmount"] = df["Quantity"] * df["Price"]



In [3]:

# ==========================================
# 2. Data Preparation
# ==========================================

# --- Convert data types ---
# 1. Convert the column to numeric values first (errors='coerce' handles any actual strings gracefully)
numeric_dates = pd.to_numeric(df["Date"], errors="coerce")

# 2. Convert Excel serial numbers to datetime using Excel's origin date
df["Date"] = pd.to_datetime(numeric_dates, unit="D", origin="1899-12-30")

# Ensure CustomerID is treated as a string
df["CustomerID"] = df["CustomerID"].astype(str)

# --- Create new features ---
# Add a new column 'Month' that extracts the month from the Date
df["Month"] = df["Date"].dt.month


# Add a new column 'RevenueCategory' based on TotalAmount
def categorize_revenue(amount):
    if amount < 50:
        return "Low"
    # Note: The cutoff for 'Medium' and 'High' was cut off in the image,
    # but you can add extra conditions here. For example:
    # elif amount < 200:
    #     return "Medium"
    else:
        return "High"  # Placeholder logic for categories above $50


df["RevenueCategory"] = df["TotalAmount"].apply(categorize_revenue)

# ==========================================
# Save the Cleaned Data
# ==========================================
df.to_csv("cleaned_retail_sales.csv", index=False)
print("Data cleaning and preparation complete!")

Data cleaning and preparation complete!


I encountered this error:
```
ValueError: time data "45029" doesn't match format "%Y-%m-%d", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
```

Removed this:
```
# Convert the Date column to a datetime object
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
```

and replaced with:
```
# 1. Convert the column to numeric values first (errors='coerce' handles any actual strings gracefully)
numeric_dates = pd.to_numeric(df["Date"], errors="coerce")

# 2. Convert Excel serial numbers to datetime using Excel's origin date
df["Date"] = pd.to_datetime(numeric_dates, unit="D", origin="1899-12-30")
```

In [4]:
# ==========================================
# View the Cleaned File
# ==========================================
# Read the cleaned file back in to verify
cleaned_df = pd.read_csv("cleaned_retail_sales.csv")

# Display basic info and the first few rows
print("\n--- Cleaned Dataset Info ---")
print(cleaned_df.info())

print("\n--- First 5 Rows of Cleaned Data ---")
# Using display(cleaned_df.head()) if you are in a Jupyter Notebook
print(cleaned_df.head())


--- Cleaned Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   TransactionID    17 non-null     object 
 1   Date             17 non-null     object 
 2   CustomerID       17 non-null     object 
 3   Product          17 non-null     object 
 4   Quantity         17 non-null     int64  
 5   Price            17 non-null     float64
 6   TotalAmount      17 non-null     float64
 7   Month            17 non-null     int64  
 8   RevenueCategory  17 non-null     object 
dtypes: float64(2), int64(2), object(5)
memory usage: 1.3+ KB
None

--- First 5 Rows of Cleaned Data ---
  TransactionID        Date CustomerID     Product  Quantity  Price  \
0       TID0001  2023-04-13   CUST0006   product B         0  32.88   
1       TID0002  2023-12-15    Unknown   product A         8  32.88   
2       TID0003  2023-09-28   CUST0001   pro